# Challenge 04: Network or Server May Fail

**PPT problem:** Network may fail  
**Technique:** Retry with exponential backoff

Temporary failures such as 500, 502, 503, 504, and timeouts are common in data ingestion. A robust client retries temporary failures with increasing wait time. It should not blindly retry permanent errors such as 400 or 401.


## Setup

Make sure the mock serving API is running:

```bash
cd day_2/day_2_1_API
python -m uvicorn mock_serving_api:app --reload --port 8011
```

If you use a different local port, update `BASE_URL` in the code cell.


## Run the Broken Program First

Do not fix the code before running it. Run it once and observe the failure signal.

Look for:

- What status code appears on the first request?
- If the client retries, does the same client_id eventually succeed?
- Which errors are safe to retry?



In [1]:
"""
Challenge 04: Flaky server.

The first two requests for each client_id fail with 503. The provider becomes
healthy after that. Fix the client so temporary failures do not stop the job.
"""

import time

import pandas as pd
import requests


BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = f"group_04_flaky_{time.time_ns()}"


def main() -> None:
    response = requests.get(
        f"{BASE_URL}/debug-lab/04-flaky/rainfall/records",
        params={
            "page": 1,
            "page_size": 10,
            "client_id": CLIENT_ID,
        },
        timeout=10,
    )
    response.raise_for_status()

    payload = response.json()
    dataframe = pd.DataFrame(payload["data"])
    print("Rainfall rows collected:", len(dataframe))
    print(dataframe[["station_id", "station_name", "rainfall_mm"]].head())


if __name__ == "__main__":
    main()


HTTPError: 503 Server Error: Service Unavailable for url: http://127.0.0.1:8011/debug-lab/04-flaky/rainfall/records?page=1&page_size=10&client_id=group_04_flaky_1782181188677775000

## Diagnose

Write short answers before changing the code.

1. What failed: HTTP request, Python processing, or data quality check?
2. What exact error/status/message did you see?
3. Which status codes are temporary enough to retry safely?
4. Which technique from the PPT table should fix it?


In [ ]:
# Notes
# 1. http request
# 2. 503 - service unavailable
# 3. 503 - retry with backoff  (Temporary 503 errors are not the same as bad data; the client should retry safely.)
# 4.


## Where to Look for the Fix

Use the API docs, the observed failure, and the clues below to decide what to change.

Primary place to check: `http://127.0.0.1:8011/docs`, then find `/debug-lab/04-flaky/rainfall/records`.

Use these clues:

1. Open `/docs` and inspect what this endpoint can return.
2. Temporary 503 errors are not the same as bad data; the client should retry safely.
3. Use a small retry loop with a short delay, then keep the successful response.

After that, use the success condition below to check whether your repaired code is good enough.


## Repair Workspace

Copy the broken code here and edit it until it works.

Success condition: The notebook should collect rainfall rows after retrying temporary 503 errors.


In [2]:
# Paste the broken code here, then repair it.
# Start by checking /docs for the endpoint contract and rereading the error output above.
# Stop when the success condition in the previous markdown cell is satisfied.

"""
Challenge 04: Flaky server.

The first two requests for each client_id fail with 503. The provider becomes
healthy after that. Fix the client so temporary failures do not stop the job.
"""

import time

import pandas as pd
import requests


BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = f"group_04_flaky_{time.time_ns()}"

def get_with_retry(url, params=None, max_attempts=5, timeout_sec = 5, base_sleep=1):
    TEMPORARY_STATUS_CODES = {429, 500, 502, 503, 504}
    # Try the same request multiple times before giving up.
    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}/{max_attempts}: sending request to the API")
        try:
            response = requests.get(url, params=params, timeout=timeout_sec)
            print(f"Received status code: {response.status_code}")

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", base_sleep))
                print(f"Request was rate limited. Waiting {retry_after}s before retrying")
                time.sleep(retry_after)
                continue

            if response.status_code in TEMPORARY_STATUS_CODES:
                wait_seconds = base_sleep * (2 ** (attempt - 1))
                print(f"Temporary API error: HTTP {response.status_code}")
                print(f"Retrying after {wait_seconds}s")
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            print("Request successful")
            return response
        except requests.Timeout:
            wait_seconds = base_sleep * (2 ** (attempt - 1))
            print("Request timed out")
            print(f"Retrying after {wait_seconds}s")
            time.sleep(wait_seconds)

    print("No retry attempts left")
    raise RuntimeError(f"Request failed after {max_attempts} attempts: {url}")


def main() -> None:
    response = get_with_retry(
        f"{BASE_URL}/debug-lab/04-flaky/rainfall/records",
        params={
            "page": 1,
            "page_size": 10,
            "client_id": CLIENT_ID,
        },
        timeout_sec=10,
    )

    payload = response.json()
    dataframe = pd.DataFrame(payload["data"])
    print("Rainfall rows collected:", len(dataframe))
    print(dataframe[["station_id", "station_name", "rainfall_mm"]].head())


if __name__ == "__main__":
    main()


Attempt 1/5: sending request to the API
Received status code: 503
Temporary API error: HTTP 503
Retrying after 1s
Attempt 2/5: sending request to the API
Received status code: 503
Temporary API error: HTTP 503
Retrying after 2s
Attempt 3/5: sending request to the API
Received status code: 200
Request successful
Rainfall rows collected: 10
  station_id           station_name  rainfall_mm
0       S218  Bukit Batok Street 34            0
1       S219      Yio Chu Kang Road            0
2       S216   Ang Mo Kio Avenue 10            0
3       S217       Bishan Street 13            0
4       S214            Tanjong Rhu            0
